# TP Final - Visión Artificial: del tablero físico al FEN y a la mejor jugada

A partir de una foto de un tablero de ajedrez y de la **dirección hacia la que avanzan las blancas**, el pipeline:

1. detecta las 4 esquinas del tablero con un modelo YOLO,
2. calcula una homografía para rectificar el tablero a un cuadrado de 800x800 px,
3. detecta las piezas con un segundo modelo YOLO y las asigna a celdas,
4. normaliza la orientación y genera el **FEN**.

Luego el script local `chess_cli.py` consulta la API de Lichess para obtener la siguiente jugada óptima.

## Configuración previa (leer antes de ejecutar)

**1. API key de Roboflow.** Entrá a Roboflow, andá a *Settings -> API Keys* y copiá la clave **privada**.
En Colab abrí el panel de claves (icono de llave), creá un secreto llamado `ROBOFLOW_API_KEY`, pegá la clave y activá **Acceso desde el notebook**.

**2. Número de versión de cada dataset.** Abrí la página del dataset en [Roboflow Universe](https://universe.roboflow.com) y mirá la pestaña *Dataset* / *Versions*: ahí figura el número de versión disponible (por ejemplo `v3`). Usá ese número en la variable `VERSION`.

**3. Si la descarga directa falla por permisos.** Algunos datasets no permiten descargar vía API. En ese caso:
*Fork Dataset* a tu propio workspace -> *Generate -> Create New Version* -> actualizá `WORKSPACE`, `PROJECT` y `VERSION` con los de tu copia.

**Datasets usados**
- Esquinas: `robotics-project-02zwj/chess-corner-points`
- Piezas: `utkarsh-srivastava-3n6fv/chess-pieces-pokxh`

In [ ]:
!pip install -q roboflow ultralytics opencv-python-headless python-chess

## 1. Setup: GPU y API key

In [ ]:
import subprocess
import torch

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
else:
    print("ATENCION: no hay GPU. En Colab: Entorno de ejecución -> Cambiar tipo -> GPU.")

In [ ]:
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
    print("API key leída desde Colab Secrets.")
except Exception as e:
    from getpass import getpass
    print(f"No se pudo leer el secreto ({type(e).__name__}). Ingresala manualmente.")
    ROBOFLOW_API_KEY = getpass("Roboflow API key: ")

In [ ]:
from roboflow import Roboflow

# Opción A (descarga directa): usar workspace/project originales del Universe.
# Opción B (fork): si falla por permisos, forkeá el dataset, generá una versión y
#                  reemplazá WORKSPACE / PROJECT / VERSION por los de tu workspace.
DATASETS = {
    "corners": {"WORKSPACE": "robotics-project-02zwj",      "PROJECT": "chess-corner-points", "VERSION": 1},
    "pieces":  {"WORKSPACE": "utkarsh-srivastava-3n6fv",     "PROJECT": "chess-pieces-pokxh",  "VERSION": 1},
}

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
rutas = {}
for nombre, cfg in DATASETS.items():
    proyecto = rf.workspace(cfg["WORKSPACE"]).project(cfg["PROJECT"])
    ds = proyecto.version(cfg["VERSION"]).download("yolov11", location=f"/content/datasets/{nombre}")
    rutas[nombre] = ds.location
    print(nombre, "->", ds.location)

In [ ]:
import os

for nombre, ruta in rutas.items():
    print("=" * 60, nombre, ruta)
    for raiz, dirs, files in os.walk(ruta):
        nivel = raiz[len(ruta):].count(os.sep)
        if nivel <= 2:
            print("  " * nivel + os.path.basename(raiz) + "/", f"({len(files)} archivos)" if files else "")
    print(open(os.path.join(ruta, "data.yaml")).read())

## 2. Limpieza de labels del dataset de piezas

El `data.yaml` del dataset de piezas trae como `names` texto del README de Roboflow en lugar de nombres de piezas. Se mapean a mano a las 12 clases canónicas `{white,black}_{pawn,knight,bishop,rook,queen,king}`, se **reescriben los `.txt`** con los nuevos IDs y se genera un `data_clean.yaml`. Las clases sin mapeo se descartan y se reportan.

In [ ]:
import yaml

RUTA_PIEZAS = rutas["pieces"]
with open(f"{RUTA_PIEZAS}/data.yaml") as f:
    yaml_crudo = yaml.safe_load(f)
nombres_crudos = yaml_crudo["names"]
if isinstance(nombres_crudos, dict):
    nombres_crudos = [nombres_crudos[k] for k in sorted(nombres_crudos)]
print("Nombres crudos del data.yaml:")
for i, n in enumerate(nombres_crudos):
    print(f"  {i}: {n!r}")

In [ ]:
CLASES = [f"{c}_{p}" for c in ("white", "black") for p in ("pawn", "knight", "bishop", "rook", "queen", "king")]
ID_CLASE = {n: i for i, n in enumerate(CLASES)}

### Mapeo manual de las clases

Ningún nombre del `data.yaml` dice qué pieza es, así que el mapeo se define a mano en `MAPEO_MANUAL`: la clave es el nombre **exacto** que aparece en el yaml y el valor es la clase canónica real. Verificá en Roboflow qué pieza representa cada clase y corregí la tabla si hace falta.

In [ ]:
# Clave: nombre EXACTO que aparece en el data.yaml. Valor: clase canónica real de esa entrada.
MAPEO_MANUAL = {
    "- annotate- and create datasets": "white_king", # "white_pawn",
    "- collaborate with your team on computer vision projects": "black_queen", # "white_knight",
    "- collect - organize images": "black_rook", # "white_bishop",
    "- export- train- and deploy computer vision models": "white_knight", # "white_rook",
    "- understand and search unstructured image data": "white_bishop", # "white_queen",
    "- use active learning to improve your dataset over time": "white_pawn", # "white_king",
    "------------------------------": "black_king", # "black_pawn",
    "For state of the art Computer Vision training notebooks you can use with this dataset-": "white_queen", # "black_knight",
    "Roboflow is an end-to-end computer vision platform that helps you": "black_pawn", # "black_bishop",
    "This dataset was exported via roboflow.com on November 7- 2024 at 6-44 AM GMT": "black_knight", # "black_rook",
    "fd - v2 2023-04-14 6-55pm": "black_bishop", # "black_queen",
    "visit https---github.com-roboflow-notebooks": "white_rook" # "black_king",
}

assert all(v in ID_CLASE for v in MAPEO_MANUAL.values()), "MAPEO_MANUAL tiene clases canónicas inválidas"
assert len(set(MAPEO_MANUAL.values())) == len(MAPEO_MANUAL), "MAPEO_MANUAL asigna la misma clase a dos entradas"

In [ ]:
from pathlib import Path

mapa_ids = {}
descartadas = []
for i, n in enumerate(nombres_crudos):
    canon = MAPEO_MANUAL.get(str(n).strip())
    if canon is None:
        descartadas.append((i, n))
    else:
        mapa_ids[i] = ID_CLASE[canon]
        print(f"  {i:2d} {n[:45]!r:50} -> {canon} ({ID_CLASE[canon]})")
print("\nClases descartadas (sin mapeo):", descartadas or "ninguna")
assert mapa_ids, "Ninguna clase pudo mapearse: revisá MAPEO_MANUAL y los nombres crudos impresos arriba."

# Respaldo de las labels originales (una sola vez): la limpieza SIEMPRE lee del respaldo,
# así se puede re-ejecutar esta celda sin corromper nada.
import shutil
from collections import Counter

for split in ("train", "valid", "test"):
    carpeta = Path(RUTA_PIEZAS) / split / "labels"
    respaldo = Path(RUTA_PIEZAS) / split / "labels_orig"
    if carpeta.exists() and not respaldo.exists():
        shutil.copytree(carpeta, respaldo)

# Diagnóstico del dataset crudo: cuántas anotaciones hay y qué IDs usan
ids_crudos = Counter()
for txt in (Path(RUTA_PIEZAS) / "train" / "labels_orig").glob("*.txt"):
    for linea in txt.read_text().splitlines():
        if linea.strip():
            ids_crudos[int(float(linea.split()[0]))] += 1
print("IDs en las labels crudas de train:", dict(sorted(ids_crudos.items())) or "NINGUNA (labels vacías)")
assert ids_crudos, ("Las labels originales están vacías: el dataset se descargó sin anotaciones. "
                    "Revisá VERSION / el fork y volvé a descargar.")
sin_nombre = sorted(set(ids_crudos) - set(range(len(nombres_crudos))))
if sin_nombre:
    print("ATENCIÓN: hay IDs en las labels que no existen en data.yaml:", sin_nombre)

stats = {"archivos": 0, "lineas_ok": 0, "lineas_descartadas": 0}
for split in ("train", "valid", "test"):
    origen = Path(RUTA_PIEZAS) / split / "labels_orig"
    destino = Path(RUTA_PIEZAS) / split / "labels"
    if not origen.exists():
        print("(sin split)", split)
        continue
    for txt in origen.glob("*.txt"):
        nuevas = []
        for linea in txt.read_text().splitlines():
            partes = linea.split()
            if not partes:
                continue
            viejo = int(float(partes[0]))
            if viejo in mapa_ids:
                nuevas.append(" ".join([str(mapa_ids[viejo])] + partes[1:]))
                stats["lineas_ok"] += 1
            else:
                stats["lineas_descartadas"] += 1
        (destino / txt.name).write_text("\n".join(nuevas) + ("\n" if nuevas else ""))
        stats["archivos"] += 1
print("Resumen de reescritura:", stats)
assert stats["lineas_ok"] > 0, "La limpieza no dejó ninguna anotación: revisá MAPEO_MANUAL y los IDs de arriba."

# data.yaml limpio
yaml_limpio = {
    "path": RUTA_PIEZAS,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(CLASES),
    "names": {i: n for i, n in enumerate(CLASES)},
}
YAML_PIEZAS = f"{RUTA_PIEZAS}/data_clean.yaml"
with open(YAML_PIEZAS, "w") as f:
    yaml.safe_dump(yaml_limpio, f, sort_keys=False)

# Validación: ningún ID fuera de rango
fuera = 0
for split in ("train", "valid", "test"):
    for txt in (Path(RUTA_PIEZAS) / split / "labels").glob("*.txt"):
        for linea in txt.read_text().splitlines():
            if linea.strip() and not (0 <= int(linea.split()[0]) < len(CLASES)):
                fuera += 1
assert fuera == 0, f"Hay {fuera} labels con ID fuera de rango"
print("Validación OK: todos los IDs están en [0, 11].")
print(open(YAML_PIEZAS).read())

## 3. Entrenamiento

Se entrenan **dos modelos independientes** a partir de `yolo11n.pt`: uno para esquinas (1 clase) y otro para piezas (12 clases). Ajustá `EPOCHS` y `BATCH` según el tiempo y la memoria de la GPU.

In [ ]:
from ultralytics import YOLO

EPOCHS = 100
BATCH = 16
PATIENCE = 20
SEED = 0
IMGSZ = 640

def entrenar(yaml_path, nombre):
    modelo = YOLO("yolo11n.pt")
    return modelo.train(
        data=yaml_path, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0,
        patience=PATIENCE, seed=SEED, amp=True,
        project="/content/runs", name=nombre, exist_ok=True,
    )

YAML_ESQUINAS = f"{rutas['corners']}/data.yaml"
res_esquinas = entrenar(YAML_ESQUINAS, "corners")

In [ ]:
res_piezas = entrenar(YAML_PIEZAS, "pieces")

In [ ]:
import shutil
import pandas as pd
from IPython.display import Image, display

for nombre in ("corners", "pieces"):
    carpeta = Path("/content/runs") / nombre
    df = pd.read_csv(carpeta / "results.csv")
    df.columns = [c.strip() for c in df.columns]
    print("=" * 50, nombre)
    print(df.iloc[-1][[c for c in df.columns if "metrics" in c]])
    display(Image(filename=str(carpeta / "results.png")))
    shutil.copy(carpeta / "weights" / "best.pt", f"/content/{nombre}.pt")
    print("Guardado:", f"/content/{nombre}.pt")

In [ ]:
# Validación formal sobre el split de validación
for nombre in ("corners", "pieces"):
    m = YOLO(f"/content/{nombre}.pt")
    r = m.val(data=YAML_ESQUINAS if nombre == "corners" else YAML_PIEZAS, device=0, plots=False, verbose=False)
    print(f"{nombre}: mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}  P={r.box.mp:.3f}  R={r.box.mr:.3f}")

### Descarga de los pesos

Descargá `corners.pt` y `pieces.pt` y copialos en **`tpFinal/models/`** del repositorio (los usa `chess_cli.py`).

In [ ]:
from google.colab import files

files.download("/content/corners.pt")
files.download("/content/pieces.pt")

In [ ]:
# Opcional: copiar a Google Drive
from google.colab import drive

drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/tpFinal/models", exist_ok=True)
for nombre in ("corners", "pieces"):
    shutil.copy(f"/content/{nombre}.pt", f"/content/drive/MyDrive/tpFinal/models/{nombre}.pt")
print("Copiado a Drive: MyDrive/tpFinal/models/")

## 4. Pipeline matemático

**Homografía.** Una homografía `H` (matriz 3x3) lleva un punto de la imagen `(x, y)` a su posición `(x', y')` en el tablero rectificado, en coordenadas homogéneas:

`s·[x', y', 1]ᵀ = H·[x, y, 1]ᵀ`

donde `s` es un factor de escala. Con 4 correspondencias (las 4 esquinas de la imagen y las 4 esquinas de un cuadrado de 800x800) `H` queda determinada. Cada celda mide 100x100 px, de modo que `fila = floor(y'/100)`, `col = floor(x'/100)`.

**Punto de apoyo.** Para una pieza se usa el *bottom-center* del bbox `((x1+x2)/2, y2)`, el punto más cercano al plano del tablero; así se evita el error de paralaje de las piezas altas.

**Orden de esquinas.** TL = mínimo de `x+y`; BR = máximo de `x+y`; TR = mínimo de `y-x`; BL = máximo de `y-x`.

**Orientación.** La dirección es un *input explícito* del usuario: hacia dónde avanzan las blancas en la imagen. Se rota la matriz con `np.rot90(m, k)` (antihorario) hasta dejar las blancas abajo (fila 0 = rank 8, columna 0 = columna a):

| Dirección | Blancas en la imagen | k |
|---|---|---|
| `arriba` | abajo | 0 |
| `derecha` | a la izquierda | 1 |
| `abajo` | arriba | 2 |
| `izquierda` | a la derecha | 3 |

Una heurística (densidad de piezas blancas en las 2 filas/columnas de cada borde) sirve **solo como verificación cruzada**: si contradice al usuario emite un *warning*, no un error.

In [ ]:
import warnings
import cv2
import numpy as np
import matplotlib.pyplot as plt

LADO = 800
CELDA = LADO // 8
SIMBOLO = {
    "white_pawn": "P", "white_knight": "N", "white_bishop": "B", "white_rook": "R", "white_queen": "Q", "white_king": "K",
    "black_pawn": "p", "black_knight": "n", "black_bishop": "b", "black_rook": "r", "black_queen": "q", "black_king": "k",
}
K_POR_DIRECCION = {"arriba": 0, "derecha": 1, "abajo": 2, "izquierda": 3}


def ordenar_esquinas(pts):
    """Ordena 4 puntos como TL, TR, BR, BL.

    TL = menor x+y, BR = mayor x+y, TR = menor y-x, BL = mayor y-x.

    Args:
        pts: array (4, 2) con puntos (x, y) en cualquier orden.
    Returns:
        np.ndarray float32 (4, 2) ordenado [TL, TR, BR, BL].
    """
    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)
    suma = pts.sum(axis=1)
    dif = pts[:, 1] - pts[:, 0]
    return np.array([pts[np.argmin(suma)], pts[np.argmin(dif)], pts[np.argmax(suma)], pts[np.argmax(dif)]], dtype=np.float32)


def esquinas_desde_yolo(results):
    """Toma los centros de bbox de las 4 detecciones de mayor confianza y las ordena.

    Args:
        results: salida de `modelo.predict(...)` (lista de Results).
    Returns:
        np.ndarray (4, 2) ordenado [TL, TR, BR, BL].
    Raises:
        ValueError: si hay menos de 4 detecciones.
    """
    boxes = results[0].boxes
    n = 0 if boxes is None else len(boxes)
    if n < 4:
        raise ValueError(f"Se detectaron {n} esquinas; se necesitan 4. Mejorá la foto o la iluminación.")
    xyxy = boxes.xyxy.cpu().numpy()
    conf = boxes.conf.cpu().numpy()
    top = np.argsort(-conf)[:4]
    centros = np.stack([(xyxy[top, 0] + xyxy[top, 2]) / 2, (xyxy[top, 1] + xyxy[top, 3]) / 2], axis=1)
    return ordenar_esquinas(centros)


def calcular_homografia(esquinas, lado=LADO):
    """Calcula H que lleva las esquinas (TL, TR, BR, BL) al cuadrado lado x lado.

    s·[x', y', 1]^T = H·[x, y, 1]^T
    """
    dst = np.array([[0, 0], [lado, 0], [lado, lado], [0, lado]], dtype=np.float32)
    H, _ = cv2.findHomography(np.asarray(esquinas, dtype=np.float32), dst)
    if H is None:
        raise ValueError("No se pudo calcular la homografía (esquinas degeneradas).")
    return H


def mapear_piezas(detecciones, H, lado=LADO):
    """Asigna cada detección a una celda del tablero rectificado.

    Args:
        detecciones: lista de dicts {'bbox': (x1,y1,x2,y2), 'clase': str, 'conf': float}.
        H: homografía imagen -> tablero.
    Returns:
        Matriz 8x8 (lista de listas) de strings; '' si la celda está vacía.
        Ante dos piezas en la misma celda gana la de mayor confianza.
    """
    celda = lado / 8
    matriz = [[""] * 8 for _ in range(8)]
    mejor = np.zeros((8, 8))
    if not detecciones:
        return matriz
    pts = np.array([[[(d["bbox"][0] + d["bbox"][2]) / 2, d["bbox"][3]]] for d in detecciones], dtype=np.float32)
    mapeados = cv2.perspectiveTransform(pts, H).reshape(-1, 2)
    for d, (x, y) in zip(detecciones, mapeados):
        col = int(np.clip(x // celda, 0, 7))
        fila = int(np.clip(y // celda, 0, 7))
        if d["conf"] > mejor[fila, col]:
            mejor[fila, col] = d["conf"]
            matriz[fila][col] = d["clase"]
    return matriz


def _heuristica_direccion(arr):
    """Estima hacia dónde avanzan las blancas por densidad de blancas en 2 filas/cols por borde."""
    blanca = np.vectorize(lambda s: str(s).startswith("white_"))(arr)
    borde = {
        "arriba": blanca[6:, :].sum(),
        "abajo": blanca[:2, :].sum(),
        "derecha": blanca[:, :2].sum(),
        "izquierda": blanca[:, 6:].sum(),
    }
    orden = sorted(borde.items(), key=lambda kv: -kv[1])
    if orden[0][1] == 0 or orden[0][1] == orden[1][1]:
        return None
    return orden[0][0]


def normalizar_orientacion(matriz, direccion):
    """Rota la matriz para que las blancas queden abajo (fila 0 = rank 8).

    Args:
        matriz: 8x8 tal como se ve en la imagen rectificada.
        direccion: 'arriba' | 'abajo' | 'izquierda' | 'derecha' = hacia dónde avanzan las blancas
            en la imagen ('arriba' significa blancas abajo, k=0).
    Emite un warning (no un error) si la heurística de densidad contradice al usuario.
    """
    if direccion not in K_POR_DIRECCION:
        raise ValueError(f"Dirección inválida: {direccion!r}")
    arr = np.array(matriz, dtype=object)
    sugerida = _heuristica_direccion(arr)
    if sugerida is not None and sugerida != direccion:
        warnings.warn(f"La heurística sugiere '{sugerida}' pero indicaste '{direccion}'. Se respeta lo indicado.")
    return np.rot90(arr, k=K_POR_DIRECCION[direccion]).tolist()


def matriz_a_fen(matriz, turno="w"):
    """Convierte la matriz 8x8 (fila 0 = rank 8) a FEN con sufijo '{turno} - - 0 1'."""
    filas = []
    for fila in matriz:
        s, vacias = "", 0
        for celda in fila:
            if not celda:
                vacias += 1
                continue
            if vacias:
                s += str(vacias)
                vacias = 0
            s += SIMBOLO[celda]
        if vacias:
            s += str(vacias)
        filas.append(s)
    return "/".join(filas) + f" {turno} - - 0 1"

### Tests sintéticos del pipeline

In [ ]:
import chess

INICIAL = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR"
NOM = {"r": "rook", "n": "knight", "b": "bishop", "q": "queen", "k": "king", "p": "pawn"}

def matriz_inicial():
    m = [[""] * 8 for _ in range(8)]
    for i, fila in enumerate(str(chess.Board()).split("\n")):
        for j, ch in enumerate(fila.split()):
            if ch != ".":
                m[i][j] = f"{'white' if ch.isupper() else 'black'}_{NOM[ch.lower()]}"
    return m

# ordenar_esquinas
o = ordenar_esquinas([[90, 100], [10, 10], [10, 95], [100, 8]])
assert np.array_equal(o, [[10, 10], [100, 8], [90, 100], [10, 95]])

# homografía: round-trip
esq = np.array([[120, 80], [700, 120], [760, 640], [60, 600]], dtype=np.float32)
H = calcular_homografia(esq)
proy = cv2.perspectiveTransform(esq.reshape(-1, 1, 2), H).reshape(-1, 2)
assert np.allclose(proy, [[0, 0], [800, 0], [800, 800], [0, 800]], atol=1e-2)
back = cv2.perspectiveTransform(proy.reshape(-1, 1, 2).astype(np.float32), np.linalg.inv(H)).reshape(-1, 2)
assert np.allclose(back, esq, atol=0.1)

# mapear_piezas: celdas, clip y conflicto
dets = [
    {"bbox": [0, 0, 100, 50], "clase": "white_pawn", "conf": 0.9},
    {"bbox": [0, 0, 100, 60], "clase": "white_rook", "conf": 0.5},
    {"bbox": [710, 600, 790, 799], "clase": "black_king", "conf": 0.8},
    {"bbox": [-50, 0, 50, 2000], "clase": "black_queen", "conf": 0.7},
]
m = mapear_piezas(dets, np.eye(3))
assert m[0][0] == "white_pawn" and m[7][7] == "black_king" and m[7][0] == "black_queen"

# matriz_a_fen
assert matriz_a_fen(matriz_inicial()) == INICIAL + " w - - 0 1"

# normalizar_orientacion: la posición inicial rotada debe volver al mismo FEN
base = np.array(matriz_inicial(), dtype=object)
# izquierda: blancas a la derecha (rot antihorario de la base); derecha: blancas a la izquierda
for direccion, k_img in {"arriba": 0, "izquierda": 1, "abajo": 2, "derecha": -1}.items():
    imagen = np.rot90(base, k=k_img).tolist()
    with warnings.catch_warnings():
        warnings.simplefilter("error")
        fen = matriz_a_fen(normalizar_orientacion(imagen, direccion))
    assert fen.split()[0] == INICIAL, direccion
    print(f"{direccion:10} OK")
print("Todos los tests del pipeline pasaron.")

## 5. Inferencia de demostración

`imagen_a_fen(path, direccion, turno)` encadena todo el pipeline y muestra: imagen con detecciones, tablero rectificado con grilla, matriz y FEN.

In [ ]:
modelo_esquinas = YOLO("/content/corners.pt")
modelo_piezas = YOLO("/content/pieces.pt")


def imagen_a_fen(path, direccion, turno="w", mostrar=True):
    """Encadena: esquinas -> homografía -> piezas -> matriz -> orientación -> FEN.

    Args:
        path: ruta a la imagen.
        direccion: 'arriba' | 'abajo' | 'izquierda' | 'derecha' (hacia dónde avanzan las blancas).
        turno: 'w' o 'b'.
    Returns:
        str con el FEN.
    """
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)
    res_esq = modelo_esquinas.predict(img, verbose=False)
    esquinas = esquinas_desde_yolo(res_esq)
    H = calcular_homografia(esquinas)

    res_pie = modelo_piezas.predict(img, verbose=False)[0]
    dets = []
    for b in res_pie.boxes:
        nombre = res_pie.names[int(b.cls)]
        if nombre in SIMBOLO:
            dets.append({"bbox": b.xyxy[0].cpu().numpy().tolist(), "clase": nombre, "conf": float(b.conf)})
    matriz = mapear_piezas(dets, H)
    matriz_norm = normalizar_orientacion(matriz, direccion)
    fen = matriz_a_fen(matriz_norm, turno)

    if mostrar:
        anot = img.copy()
        for p in esquinas:
            cv2.circle(anot, tuple(int(v) for v in p), 12, (0, 255, 0), -1)
        for d in dets:
            x1, y1, x2, y2 = (int(v) for v in d["bbox"])
            cv2.rectangle(anot, (x1, y1), (x2, y2), (255, 0, 0), 2)
        rect = cv2.warpPerspective(img, H, (LADO, LADO))
        for i in range(9):
            cv2.line(rect, (i * CELDA, 0), (i * CELDA, LADO), (0, 255, 0), 1)
            cv2.line(rect, (0, i * CELDA), (LADO, i * CELDA), (0, 255, 0), 1)
        fig, ax = plt.subplots(1, 2, figsize=(14, 7))
        ax[0].imshow(cv2.cvtColor(anot, cv2.COLOR_BGR2RGB)); ax[0].set_title("Detecciones"); ax[0].axis("off")
        ax[1].imshow(cv2.cvtColor(rect, cv2.COLOR_BGR2RGB)); ax[1].set_title("Tablero rectificado"); ax[1].axis("off")
        plt.show()
        print("Matriz (ya orientada, blancas abajo):")
        for fila in matriz_norm:
            print(" ".join(SIMBOLO.get(c, ".") for c in fila))
        print("FEN:", fen)
        try:
            print(chess.Board(fen))
        except ValueError as e:
            print("FEN inválido:", e)
    return fen

In [ ]:
# Subir una imagen propia
from google.colab import files

subidas = files.upload()
DIRECCION = "arriba"  # hacia dónde avanzan las blancas en la imagen
for nombre in subidas:
    imagen_a_fen(nombre, DIRECCION, turno="w")

In [ ]:
# Probar con imágenes del split de test del dataset de piezas
import glob

DIRECCION_TEST = "arriba"  # ajustar según las fotos del dataset
for p in sorted(glob.glob(f"{rutas['pieces']}/test/images/*"))[:3]:
    print("#" * 60, p)
    try:
        imagen_a_fen(p, DIRECCION_TEST)
    except ValueError as e:
        print("Error:", e)

## 6. Cierre y limitaciones

- **Oclusiones:** una pieza tapada por otra (o por la mano) no se detecta y la celda queda vacía.
- **Piezas altas:** aunque se usa el bottom-center del bbox, con ángulos muy rasantes un rey o una reina pueden caer en la celda vecina.
- **Iluminación y reflejos:** sombras, brillos o baja luz degradan tanto la detección de esquinas como la de piezas.
- **Enroque y en passant:** no se pueden inferir de una sola foto; el FEN usa `- -` (sin derechos de enroque ni captura al paso), por lo que Lichess nunca sugerirá enroque.
- **Turno, reloj y número de jugada:** se asumen (`turno` lo elige el usuario; `0 1`).
- **Orientación:** depende de que el usuario indique correctamente hacia dónde avanzan las blancas; la heurística solo advierte.
- **Cloud-eval como caché:** la API de Lichess solo responde para posiciones ya analizadas (404 en caso contrario) y tiene rate limit; un error de detección suele hacer que la posición no exista en la caché.

Pesos entrenados: copiar `corners.pt` y `pieces.pt` a `tpFinal/models/` y usar `chess_cli.py`.